# 🚀 Google Colab GPU Training Bridge

Since the Google Colab file upload widget (`files.upload()`) is disabled inside VS Code's Jupyter interface, use **Google Drive** to sync your files:

### 📂 Setup Instructions:
1. Run this local terminal command in your workspace to package the project:
   `zip -r Stock_Transformer.zip src/ main.py predict.py app.py requirements.txt -x "stock_env/*"` 
2. Go to your [Google Drive](https://drive.google.com/) browser interface and upload `Stock_Transformer.zip` directly to the root of your Drive.
3. Execute the cells below in order.

In [1]:
# Mount Google Drive (this will prompt you to authorize the mount inside VS Code)
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

# Copy and unzip the code
zip_path = "/content/drive/MyDrive/Stock_Transformer.zip"
if os.path.exists(zip_path):
    shutil.copy(zip_path, "/content/Stock_Transformer.zip")
    !unzip -o -q /content/Stock_Transformer.zip -d /content/
    print("✅ Stock_Transformer code successfully unzipped!")
else:
    print(f"❌ Could not find {zip_path} in your Google Drive root. Please upload it first!")

Mounted at /content/drive
✅ Stock_Transformer code successfully unzipped!


In [3]:
# Install dependencies (PyTorch is pre-installed on Colab)
!pip install -q yfinance plotly pandas numpy scikit-learn joblib
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Device: Tesla T4


In [4]:
# Start training on the T4 GPU using a 1e-3 learning rate
!python main.py --epochs 50 --batch_size 256 --lr 1e-3 --d_model 128 --nhead 4 --num_layers 4 --dropout 0.25

2026-06-07 10:06:46.856363: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Preparing global multi-stock dataset for 41 tickers...
Fitting baseline global scalers on AAPL...
  [cache] Fetching fresh AAPL data from yfinance (period=max)...
  [cache] Saved 11462 rows to cache
  [cache] Fetching fresh ^GSPC data from yfinance (period=max)...
  [cache] Saved 24724 rows to cache
Scalers saved to models/
  [cache] Using cached AAPL data (0m old, TTL 240m)
  [cache] Using cached ^GSPC data (0m old, TTL 240m)
  AAPL: train=10834, val=252, test=357
  [cache] Fetching fresh MSFT data from yfinance (period=max)...
  [cache] Saved 10136 rows to cache
  [cache] Using cached ^GSPC data (0m old, TTL 240m)
  MSFT: train=9508, val=252, test=357
  [cache] Fetching

In [ ]:
# Run model inference to verify (match the d_model and stock_embed_dim you used during training!)
!python predict.py --ticker AAPL --d_model 128 --stock_embed_dim 32

In [ ]:
# Copy trained weights and scalers back to your Google Drive
import os
import shutil

if os.path.exists("best_model.pth"):
    shutil.copy("best_model.pth", "/content/drive/MyDrive/best_model.pth")
    print("✅ Saved best_model.pth to your Google Drive!")

if os.path.exists("models"):
    shutil.make_archive("/content/drive/MyDrive/trained_outputs", "zip", "models")
    print("✅ Saved trained_outputs.zip (scalers) to your Google Drive!")